# Notebook 2 — Extract Keypoints — IMPROVED
**Kaggle T4x2 · ~60 phút**

### Thay đổi so với v1:
- Feature `34` → **`51`**: thêm **confidence** từng keypoint
- Tính thêm **velocity** (delta giữa frame) → feature cuối cùng = `51+51 = 102`
- **Dùng thêm `fallv2`** (5 class: Blank/Stand/Fall/Lie/Likefall) — dataset lớn hơn nhiều
- `fallv2` label map: Fall+Lie+Likefall → 1,  Blank+Stand → 0
- Pose model: `yolov8n-pose` → **`yolov8s-pose`** (tốt hơn ~8%)

In [ ]:
!pip install ultralytics tqdm -q

import os, csv, json, shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
import torch

print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
BASE     = Path('/kaggle/input/datasets/nguynteincong/dataset-fall-fire/datasets')
SEQ_DIR  = BASE / 'fall' / 'sequences'
CSV_DIR  = BASE / 'fall' / 'csv'
FALLV2   = BASE / 'fallv2'
OUT_DIR  = Path('/kaggle/working/keypoints')
OUT_DIR.mkdir(exist_ok=True)

sequences = sorted([d for d in SEQ_DIR.iterdir() if d.is_dir()])
print(f'Fall sequences: {len(sequences)}')

# Kiểm tra fallv2
for split in ['train', 'val']:
    for cls in ['Blank', 'Stand', 'Fall', 'Lie', 'Likefall']:
        p = FALLV2 / split / cls
        # Đếm cả file ảnh trực tiếp lẫn subfolder
        n = len(list(p.glob('*'))) if p.exists() else 0
        print(f'  fallv2/{split}/{cls}: {n} sequences')

In [ ]:
IMG_EXTS = {'.png', '.jpg', '.jpeg'}

def extract_keypoints_from_frame(r):
    """Trả về vector 51 = 17 × (x, y, conf). Zeros nếu không detect được."""
    if r.keypoints is not None and len(r.keypoints.xyn) > 0 and r.boxes is not None and len(r.boxes) > 0:
        boxes = r.boxes.xyxy.cpu().numpy()
        areas = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
        best  = int(np.argmax(areas))
        xy    = r.keypoints.xyn[best].cpu().numpy()   # (17, 2)
        conf  = r.keypoints.conf[best].cpu().numpy()  # (17,)
        kp    = np.concatenate([xy, conf[:,None]], axis=1)  # (17, 3)
        return kp.flatten().astype(np.float32)  # 51
    return np.zeros(51, dtype=np.float32)


def extract_sequence(pose_model, frames, imgsz=416, batch_size=8):
    """Batch inference toàn bộ sequence — nhanh hơn 3-5x so với từng frame."""
    raw_kps = []
    for i in range(0, len(frames), batch_size):
        batch = [str(f) for f in frames[i:i+batch_size]]
        results = pose_model(batch, imgsz=imgsz, conf=0.3, verbose=False)
        for r in results:
            raw_kps.append(extract_keypoints_from_frame(r))
    return raw_kps


def add_velocity(kps_array):
    """
    kps_array: (T, 51)
    Trả về (T, 102) = concat(kps, velocity)
    velocity[t] = kps[t] - kps[t-1], với velocity[0] = 0
    """
    vel = np.zeros_like(kps_array)          # (T, 51)
    vel[1:] = kps_array[1:] - kps_array[:-1]
    return np.concatenate([kps_array, vel], axis=1)  # (T, 102)


def safe_tag(*parts):
    """Tạo tên file an toàn — loại bỏ ký tự đặc biệt từ tên folder."""
    import re
    return re.sub(r'[^\w]', '_', '_'.join(str(p) for p in parts))


def read_csv(csv_path):
    mapping = {}
    with open(csv_path) as f:
        for row in csv.reader(f):
            if len(row) < 3:
                continue
            try:
                mapping[int(row[0])] = float(row[2])
            except ValueError:
                continue
    return mapping


def make_frame_labels(acc_map, n_frames, thr_fall=1.5, window=20):
    labels     = np.zeros(n_frames, dtype=np.int64)
    peak_frame = max(acc_map, key=acc_map.get, default=None)
    if peak_frame is None or acc_map[peak_frame] < thr_fall:
        return labels
    start = max(0, peak_frame - window - 1)
    end   = min(n_frames, peak_frame)
    labels[start:end] = 1
    return labels


print('Hàm định nghĩa xong.')
print('Feature size: 102 (51 kp + 51 velocity)')
print('Batch inference: bật — nhanh hơn ~4x')


In [ ]:
from tqdm import tqdm

pose_model = YOLO('yolov8s-pose.pt')
all_seq_names = []

print('=== Xử lý fall/sequences ===')
for seq_folder in tqdm(sequences, desc='fall/sequences'):
    csv_name = '-'.join(seq_folder.name.split('-')[:2]) + '-data.csv'
    csv_path = CSV_DIR / csv_name

    frames = sorted(
        [f for f in seq_folder.iterdir() if f.suffix.lower() in IMG_EXTS],
        key=lambda x: x.name
    )
    if len(frames) == 0:
        continue

    # Batch inference — nhanh hơn nhiều so với từng frame
    raw_kps = extract_sequence(pose_model, frames, imgsz=416, batch_size=8)
    kps_51  = np.array(raw_kps, dtype=np.float32)     # (T, 51)
    kps_102 = add_velocity(kps_51)                     # (T, 102)
    n_frames = len(frames)

    if csv_path.exists():
        acc_map = read_csv(csv_path)
        labels  = make_frame_labels(acc_map, n_frames)
    else:
        labels = np.zeros(n_frames, dtype=np.int64)

    tag = safe_tag('fall_seq', seq_folder.name)  # tên file an toàn
    np.save(OUT_DIR / f'{tag}_kps.npy',    kps_102)
    np.save(OUT_DIR / f'{tag}_labels.npy', labels)
    tqdm.write(f'  {seq_folder.name}: frames={n_frames}  fall%={labels.mean()*100:.0f}%')
    all_seq_names.append(tag)

print(f'\nHoàn thành fall/sequences: {len(all_seq_names)} sequences')


In [ ]:
LABEL_MAP = {
    'Blank':    0,
    'Stand':    0,
    'Fall':     1,
    'Lie':      1,
    'Likefall': 1,
}

print('=== Xử lý fallv2 ===')
fallv2_names = []

for split_name in ['train', 'val']:
    for cls_name, lbl in LABEL_MAP.items():
        cls_dir = FALLV2 / split_name / cls_name
        if not cls_dir.exists():
            print(f'  Không tìm thấy: {cls_dir.relative_to(FALLV2)}')
            continue

        # fallv2 có thể có 2 cấu trúc:
        # 1. cls_dir/sub_seq_folder/*.png  (subfolder)
        # 2. cls_dir/*.png                 (ảnh trực tiếp)
        sub_dirs = sorted([d for d in cls_dir.iterdir() if d.is_dir()])
        direct_imgs = sorted([f for f in cls_dir.iterdir() if f.suffix.lower() in IMG_EXTS])

        if sub_dirs:
            # Cấu trúc subfolder — mỗi subfolder là 1 sequence
            sequences_v2 = sub_dirs
            print(f'  {split_name}/{cls_name} (label={lbl}): {len(sequences_v2)} sub-seq')
        elif direct_imgs:
            # Ảnh trực tiếp trong cls_dir — coi cả folder là 1 sequence
            sequences_v2 = [cls_dir]
            print(f'  {split_name}/{cls_name} (label={lbl}): ảnh trực tiếp ({len(direct_imgs)} frames)')
        else:
            print(f'  {split_name}/{cls_name}: trống, bỏ qua')
            continue

        for sub_seq in tqdm(sequences_v2, desc=f'{split_name}/{cls_name}', leave=False):
            frames = sorted(
                [f for f in sub_seq.iterdir() if f.suffix.lower() in IMG_EXTS],
                key=lambda x: x.name
            )
            if len(frames) < 5:
                continue

            raw_kps = extract_sequence(pose_model, frames, imgsz=416, batch_size=8)
            kps_51  = np.array(raw_kps, dtype=np.float32)
            kps_102 = add_velocity(kps_51)
            labels  = np.full(len(frames), lbl, dtype=np.int64)

            tag = safe_tag('fallv2', split_name, cls_name, sub_seq.name)
            np.save(OUT_DIR / f'{tag}_kps.npy',    kps_102)
            np.save(OUT_DIR / f'{tag}_labels.npy', labels)
            fallv2_names.append(tag)

print(f'\nHoàn thành fallv2: {len(fallv2_names)} sequences')
print(f'  label=0 (bình thường): {sum(1 for n in fallv2_names if "_Blank_" in n or "_Stand_" in n)}')
print(f'  label=1 (nguy hiểm)  : {sum(1 for n in fallv2_names if "_Fall_" in n or "_Lie_" in n or "_Likefall_" in n)}')


In [ ]:
# ── SPLIT: fall/sequences theo sequence-level (tránh data leak) ──────────────
train_fall, val_fall = train_test_split(all_seq_names, test_size=0.2, random_state=42)

# fallv2 đã có sẵn train/val trong tên → tách theo prefix
train_v2 = [n for n in fallv2_names if 'fallv2_train_' in n]
val_v2   = [n for n in fallv2_names if 'fallv2_val_'   in n]

final_split = {
    'train': train_fall + train_v2,
    'val':   val_fall   + val_v2
}

with open(OUT_DIR / 'split.json', 'w') as f:
    json.dump(final_split, f, indent=2)

print(f'Train sequences: {len(final_split["train"])}')
print(f'Val   sequences: {len(final_split["val"])}')

shutil.make_archive('/kaggle/working/keypoints', 'zip', '/kaggle/working/keypoints')
size_mb = os.path.getsize('/kaggle/working/keypoints.zip') / 1024 / 1024
print(f'\nkeypoints.zip: {size_mb:.1f} MB  → download rồi upload lên Kaggle Dataset')